# Prepping Data Challenge - Week 23 

## Import Libraries

In [1]:
import numpy as np
import pandas as pd

## Import Data

In [2]:
tenures_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_23_Managers.xlsx', sheet_name='Tenures')
salaries_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_23_Managers.xlsx', sheet_name='Salaries')
paid_df = pd.read_excel('D:01_Projekt_Portfolio/data_prepping_challenges/inputs/PD_2024_Wk_23_Managers.xlsx', sheet_name='Amount Paid')

In [3]:
tenures_df.head()

,Name,Gender,Store City,Store Area,Start Date,End Date
0,Pablo Prep,M,London,Stratford,2008-07-20,2017-03-02 00:00:00
1,Phil Down,M,London,Oxford Street,2008-07-20,2020-06-04 00:00:00
2,Beau Leon,M,Nottingham,NaN,2008-07-20,2018-09-05 00:00:00
3,Tasha Board,F,Leeds,NaN,2008-07-20,Current
4,Jewel Axis,M,Manchester,Trafford,2008-12-22,2022-01-07 00:00:00


In [4]:
salaries_df.head()

,Name,Salary
0,Pablo Prep,40000
1,PHIL DOWN,40000
2,Beau Leon,35000
3,Tasha Board,35000
4,Jewel Axis,37000


In [5]:
paid_df.head()

,Name,Amount Paid
0,Pablo Prep,341918
1,Phil Down,468822
2,Beau Leon,343288
3,Tasha Board,556068
4,Jewel Axis,500666


## Challenges

Find the `Managers` that have been *underpaid*

In [6]:
# Adjust naming format
salaries_df['Name'] = salaries_df['Name'].str.title()
tenures_df['Name'] = tenures_df['Name'].str.title()
paid_df['Name'] = paid_df['Name'].str.title()

In [7]:
# Join the data
df = salaries_df.merge(tenures_df, on='Name')
df = df.merge(paid_df, on = 'Name')

In [8]:
df.head()

,Name,Salary,Gender,Store City,Store Area,Start Date,End Date,Amount Paid
0,Pablo Prep,40000,M,London,Stratford,2008-07-20,2017-03-02 00:00:00,341918
1,Phil Down,40000,M,London,Oxford Street,2008-07-20,2020-06-04 00:00:00,468822
2,Beau Leon,35000,M,Nottingham,NaN,2008-07-20,2018-09-05 00:00:00,343288
3,Tasha Board,35000,F,Leeds,NaN,2008-07-20,Current,556068
4,Jewel Axis,37000,M,Manchester,Trafford,2008-12-22,2022-01-07 00:00:00,500666


In [9]:
assert df.shape[0] == salaries_df.shape[0]
assert df.shape[0] == tenures_df.shape[0]
assert df.shape[0] == paid_df.shape[0]

No loss of data

In [10]:
CURRENT_END_DATE = '2024-06-05 00:00:00'

In [11]:
# Insert current end date
df['End Date'] = np.where(df['End Date'] == 'Current', CURRENT_END_DATE, df['End Date'])

# Format End Date es datetime
df['End Date'] = pd.to_datetime(df['End Date'])

In [12]:
df

,Name,Salary,Gender,Store City,Store Area,Start Date,End Date,Amount Paid
0,Pablo Prep,40000,M,London,Stratford,2008-07-20,2017-03-02,341918
1,Phil Down,40000,M,London,Oxford Street,2008-07-20,2020-06-04,468822
2,Beau Leon,35000,M,Nottingham,NaN,2008-07-20,2018-09-05,343288
3,Tasha Board,35000,F,Leeds,NaN,2008-07-20,2024-06-05,556068
4,Jewel Axis,37000,M,Manchester,Trafford,2008-12-22,2022-01-07,500666
5,Phil Tervalues,46000,M,London,Wembley,2010-04-12,2024-06-05,650102
6,Shannon Entropy,37000,F,Manchester,Salford,2010-09-08,2022-04-01,419266
7,Anna Littick,40000,F,Birmingham,NaN,2012-11-05,2024-06-05,463562
8,Al Gore-Rhythm,40000,M,Manchester,Chorlton,2013-01-14,2024-06-05,455890
9,Linus Chart,50000,M,London,Richmond,2013-11-11,2019-01-01,257123


In [13]:
# Calculate Tenure in Days
df['Tenure'] = df['End Date'] - df['Start Date']

In [14]:
# Calculate Expected Salary
DAYS_IN_YEAR = 365.25
df['Expected Salary'] = (df['Salary'] / DAYS_IN_YEAR * df['Tenure'].dt.days).round().astype(int)

In [15]:
# Filter for underpaid managers
underpaid_managers = df[df['Amount Paid'] < df['Expected Salary']].copy()

In [16]:
# Remove Unnecessary Columns
underpaid_managers = underpaid_managers[['Name', 'Salary', 'Expected Salary', 'Amount Paid']]

In [17]:
underpaid_managers

,Name,Salary,Expected Salary,Amount Paid
0,Pablo Prep,40000,344641,341918
1,Phil Down,40000,474962,468822
2,Beau Leon,35000,354456,343288
5,Phil Tervalues,46000,650864,650102
6,Shannon Entropy,37000,427792,419266
13,Max Date,60000,240000,235059
15,Rowan Column,56000,121889,118394


**Note:** Results differ from results on website. Assumption about salary calculation might differ.

## Save Data

In [18]:
underpaid_managers.to_csv('D:01_Projekt_Portfolio/data_prepping_challenges/outputs/23_underpaid_managers.csv')